In [1]:
import pandas as pd
import numpy as np


# 一、导入数据

In [2]:
#导入收益率数据
return_df=pd.read_csv("datashare/return.csv")
#导入因子
factors_df=pd.read_csv("datashare/factors.csv")

# 二、合并收益率数据与因子

案例建模逻辑为使用t-1期的95个因子预测t期的月度收益率。因此，需要对日期进行处理。

In [3]:
#定义日期后移一个月的方法
def shift_mon(x):
    y=int(x[0:4])
    m=int(x[4:6])
    if m==12:
        s_y=y+1
        s_m=1
    else:
        s_y=y
        s_m=m+1
    
    if s_m<10:
        return str(s_y)+"0"+str(s_m)
    else:
        return str(s_y)+str(s_m)

In [4]:
return_df["Month"]=return_df["DATE"].apply(lambda x:str(x)[0:6])
return_df["S_Month"]=return_df["Month"].apply(lambda x:shift_mon(x))
return_df.drop(columns=['DATE',"Month"],inplace=True)

In [5]:
factors_df["S_Month"]=factors_df["DATE"].apply(lambda x:str(x)[0:6])
factors_df.drop(columns=['DATE'],inplace=True)

In [6]:
merge_df=pd.merge(return_df,factors_df,how="left")

In [7]:
#删除有缺失的数据
merge_df.dropna(inplace=True)

In [8]:
merge_df.iloc[:,5:-1]

,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,mom6m,mom12m,...,stdacc,stdcf,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade
1095569,1.380275e+06,1.098659,1.207052,0.090524,13.150607,0.037638,-0.183451,0.013889,0.077911,0.080453,...,0.275039,0.274212,7.0,0.029042,9.120652e-09,0.036364,0.021786,0.682479,1.951065,4.952669e-08
1095956,1.091022e+06,1.386242,1.921666,0.459892,13.380239,0.046012,-0.212605,-0.060606,0.166397,-0.258388,...,0.167736,0.169179,4.0,0.027045,1.080976e-08,0.042056,0.020227,0.496864,0.842553,6.344829e-08
1096142,1.311118e+05,1.754764,3.079196,-0.578620,11.092552,0.078732,-0.226722,0.122137,-0.321244,-0.090278,...,0.131004,0.128835,4.0,0.050909,1.210093e-07,0.092105,0.038279,0.616161,1.958914,3.609665e-08
1096965,5.687850e+04,1.603478,2.571143,-0.133013,9.233055,0.057820,-0.137648,0.012048,-0.140827,-0.138646,...,0.074168,0.064018,4.0,0.029086,4.226283e-07,0.061728,0.026198,0.811875,0.906200,8.246129e-08
1097412,2.643060e+05,1.284910,1.650993,0.134888,10.764984,0.042641,-0.203471,0.043478,-0.137955,-0.340849,...,0.110375,0.093103,1.0,0.020830,6.917992e-08,0.031447,0.015929,0.753015,0.906291,8.448533e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4038881,4.216952e+04,0.399769,0.159815,-0.026358,12.204948,0.085337,1.055196,-0.064815,-0.217391,-0.406593,...,0.238017,0.191709,0.0,0.044274,3.447987e-08,0.057692,0.023206,0.439656,5.404625,8.000931e-09
4038899,9.775939e+05,1.120539,1.255608,0.587122,13.505476,0.040518,0.186068,0.100460,0.123015,-0.271487,...,0.298482,0.323707,6.0,0.029301,4.457926e-09,0.031106,0.016967,0.639303,3.489217,1.836619e-08
4038902,5.378235e+06,1.603873,2.572409,2.706046,18.027578,0.096099,0.624630,-0.103853,2.300000,2.069809,...,0.228959,0.230257,4.0,0.089204,1.923185e-10,0.177840,0.069976,0.415221,18.474254,1.779292e-09
4038911,3.951370e+05,1.315427,1.730349,0.557775,12.421699,0.042992,0.624630,0.076239,0.189992,-0.139706,...,0.065740,0.062896,6.0,0.027122,1.144700e-08,0.045157,0.017761,0.718768,3.575852,1.992637e-08


# 三、划分训练集与测试集

In [9]:
x_sampling=merge_df.iloc[:,5:-1]
y_sampling=merge_df.iloc[:,1]

In [10]:
# 将数据划分成训练集和测试集
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_sampling, y_sampling, test_size=0.3, random_state=1)

# 四、数据标准化

In [11]:
from sklearn.preprocessing import StandardScaler
x_train = StandardScaler().fit_transform(x_train)
x_test = StandardScaler().fit_transform(x_test)

# 五、机器学习模型案例演示

各个算法参数需要根据需要进行调参，本案例均使用默认参数。

### PLS

In [12]:
from sklearn.cross_decomposition import PLSRegression
# 以默认参数建立PLS模型
pls2 = PLSRegression(n_components=2)
#拟合模型
pls2.fit(x_train, y_train)
#预测
y_pred = pls2.predict(x_test)
#报告预测值
print(y_pred)

[[ 0.05607503]
 [ 0.14456812]
 [-0.12506429]
 ...
 [-0.00516741]
 [ 0.21096288]
 [-0.21192736]]


### Ridge

In [13]:
from sklearn.linear_model import Ridge
# 以默认参数建立岭回归
rdg = Ridge(alpha=0.5)
#拟合模型
rdg.fit(x_train, y_train)
#预测
y_pred = rdg.predict(x_test)
#报告预测值
print(y_pred)

[ 0.03430906  0.1729199  -0.06811233 ...  0.01253715  0.27601138
 -0.18141419]


### LASSO

In [14]:
from sklearn.linear_model import Lasso
# 以默认参数建立LASSO模型
lasso = Lasso(alpha=0.1)
#拟合模型
lasso.fit(x_train, y_train)
#预测
y_pred = lasso.predict(x_test)
#报告预测值
print(y_pred)

[ 0.00124557  0.06545102 -0.01953501 ... -0.01014966  0.10529027
 -0.07874699]


## ElasticNet

In [15]:
from sklearn.linear_model import ElasticNet
# 以默认参数建立ElasticNet模型
EN = ElasticNet(random_state=0)
#拟合模型
EN.fit(x_train, y_train)
#预测
y_pred = EN.predict(x_test)
#报告预测值
print(y_pred)

[-0.01462682 -0.01462682 -0.01462682 ... -0.01462682 -0.01462682
 -0.01462682]


## GBRT

In [16]:
from sklearn.ensemble import GradientBoostingRegressor
# 以默认参数建立GBRT模型
GBRT = GradientBoostingRegressor(random_state=0)
#拟合模型
GBRT.fit(x_train, y_train)
#预测
y_pred = GBRT.predict(x_test)
#报告预测值
print(y_pred)

[ 0.02978865  0.16582302 -0.04066464 ...  0.00983504  0.26067985
 -0.14655009]


## RF

In [17]:
from sklearn.ensemble import RandomForestRegressor
# 以默认参数建立RF模型
RF = RandomForestRegressor(max_depth=2, random_state=0)
#拟合模型
RF.fit(x_train, y_train)
#预测
y_pred = RF.predict(x_test)
#报告预测值
print(y_pred)

[-0.03802414  0.12381943 -0.03802414 ... -0.03802414  0.12381943
 -0.22830064]


# 7.EN-ANN

In [18]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout

2023-10-15 13:38:40.134506: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [19]:
#构建人工神经网络模型
model = Sequential()
#使用 add() 函数将层添加到分类器中。其中，add函数的参数是 Dense() 函数，该函数又具有以下参数：
###Units: 定义了该特定层中的节点或神经元的数量。这里在隐藏层中设置30个神经元。
###Kernel_initializer：这定义了隐藏层中不同神经元权重的起始值。这里将其定义为“均匀”，这意味着权重将使用来自均匀分布的值进行初始化。
###Activation:这是特定隐藏层中神经元的激活函数。在这里，将函数定义为单元函数或“relu”。
###Input_dim：定义隐藏层的输入数量，将此值定义为等于输入特征数据集的列数。后续层不需要此参数，因为模型将知道前一层产生了多少输出。
model.add(Dense(units =26, activation = 'relu', input_dim = 94))
#构建输出层。其中，神经元为1，权重为均匀，激活函数选择为 Sigmoid函数。
model.add(Dense(units = 1, kernel_initializer = 'uniform'))
#编译分类器。其中，Optimizer（优化器）：优化器被选为“adam”，它是随机梯度下降的扩展。Loss（损失函数）：这定义了在训练期间要优化的损失。我们将此损失定义为均方误差。etrics（指标）：这定义了模型在测试和训练阶段要评估的指标列表。我们选择准确性作为我们的评估指标。
model.compile(optimizer = 'adam', loss = 'mean_squared_error', metrics = ['accuracy'])
#拟合模型
model.fit(x_train, y_train, batch_size =5, epochs = 10)
#预测
y_pred = model.predict(x_test)
#报告预测值
print(y_pred)

Epoch 1/10


2023-10-15 13:38:48.661860: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


27865/27865 [==============================] - 10s 352us/step - loss: 0.0048 - accuracy: 1.4355e-05
Epoch 2/10
27865/27865 [==============================] - 10s 350us/step - loss: 0.0039 - accuracy: 1.4355e-05
Epoch 3/10
27865/27865 [==============================] - 10s 350us/step - loss: 0.0036 - accuracy: 1.4355e-05
Epoch 4/10
27865/27865 [==============================] - 10s 358us/step - loss: 0.0035 - accuracy: 1.4355e-05
Epoch 5/10
27865/27865 [==============================] - 10s 349us/step - loss: 0.0034 - accuracy: 1.4355e-05
Epoch 6/10
27865/27865 [==============================] - 10s 350us/step - loss: 0.0034 - accuracy: 1.4355e-05
Epoch 7/10
27865/27865 [==============================] - 10s 349us/step - loss: 0.0033 - accuracy: 1.4355e-05
Epoch 8/10
27865/27865 [==============================] - 10s 349us/step - loss: 0.0033 - accuracy: 1.4355e-05
Epoch 9/10
27865/27865 [==============================] - 10s 350us/step - loss: 0.0032 - accuracy: 1.4355e-05
Epoch 10/10


# 六、报告样本外R2

In [21]:
#以神经网络模型为例，演示报告样本外R2的命令
from sklearn.metrics import r2_score
#计算r2_score
r2 = r2_score(y_test, y_pred)
#报告结果
print("R2分数为：", r2)

R2分数为： 0.9340412561184579
